In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

entity ------> care Episode service_src_id

MPB-------------

In [ ]:
--code 1
rserv.service_id as care_epi_service_id

In [ ]:
--code 2
LEFT JOIN
    silver_rdm_service rserv
        ON rserv.service_src_sys_inst_id = 'MPB001'
       AND trim(lower(rserv.service_src_name)) = trim(lower(ten.client_type))
       AND rserv.z_order_is_active = 1

WIP--------


In [ ]:
%%sql

SELECT TOP 100
    CONCAT('WIP', ah.file_number) AS care_epi_id,
    ah.file_number,
    serv.value_desc AS primary_service_requested,
    servt.description AS service_type_description,
    rserv.service_src_name,
    rserv.service_id
FROM silver_wip_activityheader ah
LEFT JOIN temp_silver_wip_activityheader_statistics serv
    ON ah.id = serv.activity_header_id
   AND serv.group_desc = 'Demographics'
   AND serv.type_desc = 'Which service did you use?'
   AND serv.choice_desc = 'Textbox'
LEFT JOIN (
    SELECT *
    FROM silver_wip_servicetype
    WHERE description NOT LIKE '%NOT IN USE%'
) servt
    ON ah.service_type_id = servt.id
LEFT JOIN silver_rdm_service rserv
    ON rserv.service_src_sys_inst_id = 'WIP001'
   AND trim(lower(rserv.service_src_name)) = trim(lower(serv.value_desc))
   AND rserv.z_order_is_active = 1
ORDER BY ah.file_number;

In [ ]:
%%sql

SELECT TOP 100
    CONCAT('WIP', ah.file_number) AS care_epi_id,
    ah.file_number,
    serv.value_desc AS primary_service_requested,
    servt.description AS service_type_description,
    rserv.service_src_name,
    rserv.service_id
FROM silver_wip_activityheader ah
LEFT JOIN temp_silver_wip_activityheader_statistics serv
    ON ah.id = serv.activity_header_id
   AND serv.group_desc = 'Demographics'
   AND serv.type_desc = 'Which service did you use?'
   AND serv.choice_desc = 'Textbox'
LEFT JOIN (
    SELECT *
    FROM silver_wip_servicetype
    WHERE description NOT LIKE '%NOT IN USE%'
) servt
    ON ah.service_type_id = servt.id
LEFT JOIN silver_rdm_service rserv
    ON rserv.service_src_sys_inst_id = 'WIP001'
   AND trim(lower(rserv.service_src_name)) = trim(lower(servt.description))
   AND rserv.z_order_is_active = 1
ORDER BY ah.file_number;

In [ ]:
%%sql

SELECT
    service_id,
    service_src_id,
    service_src_name,
    service_src_sys_inst_id,
    service_name_conformed
FROM silver_rdm_service
WHERE service_src_sys_inst_id = 'WIP001'
ORDER BY service_src_name;